<img src='https://hammondm.github.io/hltlogo1.png' style="float:right">
Davo Acevedo-Cardona<br>
Fall 2025<br>
Self-Paced Listening Project

## This code is used to extract the time in (ms):

This code calculates the duration (in milliseconds) of each `.wav` audio segment in a folder.  
After that it creates the lists that can be uploaded to a PsychoPy Experiment.  

#### Purpose
The idea of this particular part is to extract the exact duration of each audio segment to:
- control the timing between stimuli.
- log or analyze temporal aspects of the experiment.
- Add a column for the label of experiment: Diminutive (dim) / Plural (plu)
  
#### How it works
1. The code loops through all `.wav` files in a specified folder. (In this case: C:\Users\ual-laptop\Desktop\SPL_Gabi\exp_1\audio)
2. The length of each file is measured in **milliseconds (ms)** (`len(audio)`).
3. The durations are stored in a dictionary and saved into a CSV file (`durations.csv`).
4. The resulting CSVs lists will be loaded in PsychoPy to recreate the experiment.

# IMPORTS

In [ ]:
# This code: 
# 1. Extracts the duration (in milliseconds) of segmented audio files
# 2. Create a PsychoPy-compatible stimulus CSV file with counterbalancing

# Libraries
import pandas as pd, os, wave
import wave  # For reading WAV file properties (frames, sample rate)
import os    # For file and directory operations
import csv   # For saving duration data to CSV format
import re    # For parsing filename patterns

# DIM

In [ ]:
#COUNTERBALANCING:

base_dir  = #YOUR_PATH
audio_dir = #YOUR_PATH
input_file = os.path.join(base_dir, "stimuli_master_dim.csv")

# Experiment label
experiment_label = "dim"

def get_duration_ms(file_path):
    with wave.open(file_path, "rb") as w:
        return round((w.getnframes() / float(w.getframerate())) * 1000, 2)

#Load the CSV file:
df = pd.read_csv(input_file)

# detect segment columns (handles “seg1 (context)” header)
seg_cols = [c for c in df.columns if c.startswith("seg")]

print(f"Loaded {len(df)} items from {input_file}")
print("Segment columns:", seg_cols)

In [ ]:
df = pd.read_csv(input_file)

# Identify segment columns automatically
seg_cols = [c for c in df.columns if c.startswith("seg")]

# Ensure list column is string
df["list"] = df["list"].astype(str)

# Only keep lists 1 and 2
df = df[df["list"].isin(["1", "2"])]

# --------------------------------------------
# PROCESS LISTS 1 AND 2
# --------------------------------------------
for list_label, group in df.groupby("list"):

    rows = []

    for i, row in group.iterrows():

        for seg_col in seg_cols:

            # Extract segment number (1–8)
            seg_num_match = re.search(r"(\d+)", seg_col)
            segment_number = int(seg_num_match.group(1)) if seg_num_match else None

            # Extract file name
            audio_file = str(row[seg_col]).strip()

            # FIX: Ensure .wav extension exists
            if audio_file and not audio_file.lower().endswith(".wav"):
                audio_file = audio_file + ".wav"

            # Build full file path
            audio_path = os.path.join(audio_dir, audio_file)

            # Get audio duration
            dur_ms = get_duration_ms(audio_path)

            # Append row
            rows.append({
                "audio_file": audio_file,
                "duration_ms": dur_ms,
                "segment_number": segment_number,
                "item_id": f"item_{i+1}d",
                "genero": row["genero"],
                "fonologia": row["fonologia"],
                "list": list_label,
                "spl_exp": experiment_label
            })

    # Convert list rows to DataFrame
    out = pd.DataFrame(rows)

    # Save output file
    out_name = f"spl_stim_dim_{list_label}.csv"
    out_path = os.path.join(base_dir, out_name)
    out.to_csv(out_path, index=False)

    print(f"Saved {out_name} with {len(out)} trials")

print("Processing complete.")


# PLU

In [ ]:
#COUNTERBALANCING FOR PLURAL:

base_dir  = #YOUR_PATH
audio_dir = #YOUR_PATH
input_file = os.path.join(base_dir, "stimuli_master_plu.csv")

# Experiment label
experiment_label = "plu"

def get_duration_ms(file_path):
    with wave.open(file_path, "rb") as w:
        return round((w.getnframes() / float(w.getframerate())) * 1000, 2)

#Load the CSV file:
df = pd.read_csv(input_file)

# detect segment columns (handles “seg1 (context)” header)
seg_cols = [c for c in df.columns if c.startswith("seg")]

print(f"Loaded {len(df)} items from {input_file}")
print("Segment columns:", seg_cols)

In [ ]:
df = pd.read_csv(input_file)

# Identify segment columns automatically
seg_cols = [c for c in df.columns if c.startswith("seg")]

# Ensure list column is string
df["list"] = df["list"].astype(str)

# Only keep lists 1 and 2
df = df[df["list"].isin(["1", "2"])]

# --------------------------------------------
# PROCESS LISTS 1 AND 2
# --------------------------------------------
for list_label, group in df.groupby("list"):

    rows = []

    for i, row in group.iterrows():

        for seg_col in seg_cols:

            # Extract segment number (1–8)
            seg_num_match = re.search(r"(\d+)", seg_col)
            segment_number = int(seg_num_match.group(1)) if seg_num_match else None

            # Extract file name
            audio_file = str(row[seg_col]).strip()

            # FIX: Ensure .wav extension exists
            if audio_file and not audio_file.lower().endswith(".wav"):
                audio_file = audio_file + ".wav"

            # Build full file path
            audio_path = os.path.join(audio_dir, audio_file)

            # Get audio duration
            dur_ms = get_duration_ms(audio_path)

            # Append row
            rows.append({
                "audio_file": audio_file,
                "duration_ms": dur_ms,
                "segment_number": segment_number,
                "item_id": f"item_{i+1}p",
                "spa_sniy": row["spa_sniy"],
                "que_ykuna": row["que_ykuna"],
                "list": list_label,
                "spl_exp": experiment_label
            })

    # Convert list rows to DataFrame
    out = pd.DataFrame(rows)

    # Save output file
    out_name = f"spl_stim_plu_{list_label}.csv"
    out_path = os.path.join(base_dir, out_name)
    out.to_csv(out_path, index=False)

    print(f"Saved {out_name} with {len(out)} trials")

print("Processing complete.")

# MERGE

In [ ]:
import pandas as pd
import re

# ------------------------------------------------------
# Load your four input CSVs
# ------------------------------------------------------
dim1 = pd.read_csv("spl_stim_dim_1.csv")
dim2 = pd.read_csv("spl_stim_dim_2.csv")
plu1 = pd.read_csv("spl_stim_plu_1.csv")
plu2 = pd.read_csv("spl_stim_plu_2.csv")

# ------------------------------------------------------
# Helper functions
# ------------------------------------------------------

def extract_num(x):
    """Extract item number from filenames like d_12a_seg3.wav."""
    m = re.search(r"_(\d+)[ab]_seg", x)
    return int(m.group(1)) if m else None

def extract_letter(x):
    """Extract 'a' or 'b' from filenames like d_12a_seg3.wav."""
    m = re.search(r"_(\d+)([ab])_seg", x)
    return m.group(2) if m else None

def sort_list(df, pattern):
    """
    pattern = 'ab' → odd=a, even=b
    pattern = 'ba' → odd=b, even=a
    """
    df = df.copy()
    df["item_num"] = df["audio_file"].apply(extract_num)
    df["item_letter"] = df["audio_file"].apply(extract_letter)
    
    # Enforce the pattern
    expected_letter = []
    for n in df["item_num"]:
        if pattern == "ab":
            expected_letter.append("a" if n % 2 == 1 else "b")
        else:  # pattern == "ba"
            expected_letter.append("b" if n % 2 == 1 else "a")
    
    df = df[df["item_letter"] == expected_letter]
    
    # Sort correctly
    df = df.sort_values(by=["item_num", "segment_number"])
    
    return df.drop(columns=["item_num", "item_letter"])


# ------------------------------------------------------
# Build final lists
# ------------------------------------------------------

# LIST 1 → DIM follows a,b,a,b… and PLU follows a,b,a,b…
final_list_1 = pd.concat([
    sort_list(dim1, "ab"),
    sort_list(plu1, "ab")
], axis=0)

# LIST 2 → DIM follows b,a,b,a… and PLU follows b,a,b,a…
final_list_2 = pd.concat([
    sort_list(dim2, "ba"),
    sort_list(plu2, "ba")
], axis=0)

# ------------------------------------------------------
# Save output
# ------------------------------------------------------
final_list_1.to_csv("spl_stim_exp1_1.csv", index=False)
final_list_2.to_csv("spl_stim_exp1_2.csv", index=False)

print("FINISHED: spl_stim_exp1_1.csv and spl_stim_exp1_2.csv created correctly.")


# PRUEBA 

In [ ]:
base_dir = #YOUR_PATH
audio_dir = os.path.join(base_dir, "audio")

input_file = os.path.join(base_dir, "stimuli_master_prueba.csv")
output_file = os.path.join(base_dir, "spl_stim_prueba.csv")

experiment_label = "prueba"

# --------------------------------------------
# Function to compute WAV duration in ms
# --------------------------------------------
def get_duration_ms(file_path):
    if not os.path.exists(file_path):
        print(f"WARNING: Missing file → {file_path}")
        return None
    
    with wave.open(file_path, "rb") as w:
        return round((w.getnframes() / float(w.getframerate())) * 1000, 2)

# --------------------------------------------
# Load master CSV
# --------------------------------------------
df = pd.read_csv(input_file)
df.columns = df.columns.str.strip()   # remove trailing spaces

# Identify segment columns
seg_cols = [c for c in df.columns if c.startswith("seg")]

rows = []

# --------------------------------------------
# Process all items in the prueba list
# --------------------------------------------
for i, row in df.iterrows():
    for seg_col in seg_cols:

        # Extract segment number
        seg_num_match = re.search(r"(\d+)", seg_col)
        segment_number = int(seg_num_match.group(1)) if seg_num_match else None

        # Get audio filename
        audio_file = str(row[seg_col]).strip()

        # Ensure .wav extension
        if audio_file and not audio_file.lower().endswith(".wav"):
            audio_file = audio_file + ".wav"

        # Full audio path
        audio_path = os.path.join(audio_dir, audio_file)

        # Compute duration
        dur_ms = get_duration_ms(audio_path)

        # Append row
        rows.append({
            "audio_file": audio_file,
            "duration_ms": dur_ms,
            "segment_number": segment_number,
            "item_id": f"item_prue{i+1}",
            "list": 1,
            "spl_exp": experiment_label
        })

# Convert to DataFrame
out = pd.DataFrame(rows)

# Sort to ensure proper ordering: item_prue1 seg1→7, then item_prue2, etc.
out = out.sort_values(["item_id", "segment_number"]).reset_index(drop=True)

# Save CSV
out.to_csv(output_file, index=False, encoding="utf-8-sig")

print(f"Saved prueba stimulus file: {output_file}")